# The ReAct Loop

**Prerequisite:** finish `01_tool_calling.ipynb` first. You should already know what `tool_calls` and `ToolMessage` look like.

## What is ReAct?

ReAct stands for **Reason + Act**. It is the original prompting technique from the 2022 paper by Yao et al. (https://arxiv.org/abs/2210.03629) that made LLMs into agents.

The idea is brutally simple. You ask the model to follow a fixed text format on every turn:

```
Thought: ... (the model reasons in plain English)
Action: tool_name
Action Input: the input to the tool
Observation: ... (you paste the tool's output back in)
Thought: ...
Action: ...
Action Input: ...
Observation: ...
... (repeat) ...
Thought: I now know the final answer
Final Answer: ...
```

That is it. The whole agent is a string template. The model writes `Thought` and `Action`, you parse the text, run the tool, paste the `Observation` back, and the model continues.

## Why does this matter for the workshop?

Tool calling (notebook 1) is what modern OpenAI/Anthropic models do natively. ReAct is what people did *before* native tool calling existed, and it still works on any text-completion model. Showing it makes one thing clear: **agents are not a feature of the model. They are a pattern of prompting and looping.**

We will fetch the original community ReAct prompt from LangChain Hub (the one Harrison Chase popularised), look at it, and run it by hand.

## 0. Install

In [ ]:
# Dependencies are already declared in pyproject.toml and installed via uv add.
# If you are using the project's uv venv as kernel, imports below just work.
# uv-created venvs do NOT ship pip, so %pip install fails. Use uv pip:
!uv pip install -q "langchain>=1.0" "langchain-openai>=1.0" "langchain-core>=1.0" "langchain-community>=0.4" "langsmith>=0.4" requests python-dotenv

# On a regular venv with pip, swap to:
# %pip install -q "langchain>=1.0" "langchain-openai>=1.0" "langchain-core>=1.0" "langchain-community>=0.4" "langsmith>=0.4" requests python-dotenv

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"

## 1. Pull the original ReAct prompt from LangChain Hub

`hwchase17/react` is the prompt published by Harrison Chase, the creator of LangChain. It is the canonical ReAct prompt that thousands of tutorials reference.

We pull it and print it so we can read what is actually in it.

In [ ]:
# In langchain 1.x, `from langchain import hub` was removed.
# The replacement is langsmith.Client.pull_prompt().
# Public prompts must be pulled with dangerously_pull_public_prompt=True
# because they may contain serialized LangChain objects.
from langsmith import Client

client = Client()
react_prompt = client.pull_prompt(
    "hwchase17/react",
    include_model=False,
    dangerously_pull_public_prompt=True,
)

print("Prompt input variables:", react_prompt.input_variables)
print()
print("=" * 60)
print("THE FULL PROMPT TEMPLATE")
print("=" * 60)
print(react_prompt.template)

**Read that prompt carefully.** It tells the model exactly what format to follow:

- `Question: ...`
- `Thought: ...`
- `Action: ...` (one of the tool names)
- `Action Input: ...`
- `Observation: ...`
- ... (repeats) ...
- `Thought: I now know the final answer`
- `Final Answer: ...`

There is no special API. There are no tool-call JSON objects. It is just plain text the model is asked to follow. Our job is to:

1. Fill in the placeholders (`{tools}`, `{tool_names}`, `{input}`, `{agent_scratchpad}`).
2. Send it to the LLM.
3. **Stop** the LLM as soon as it writes `Observation:` (because that line should come from us, not the model).
4. Parse the `Action` and `Action Input`, run the tool, paste the result as `Observation`, and loop.

We will build that loop by hand.

## 2. Define tools (same shape as notebook 1, but we describe them as plain text)

In [ ]:
import requests
from langchain_core.tools import tool

@tool
def add(expression: str) -> str:
    """Adds two integers. Input must be two integers separated by a comma, like '17, 25'."""
    a, b = [int(x.strip()) for x in expression.split(",")]
    return str(a + b)

@tool
def multiply(expression: str) -> str:
    """Multiplies two integers. Input must be two integers separated by a comma, like '6, 7'."""
    a, b = [int(x.strip()) for x in expression.split(",")]
    return str(a * b)

@tool
def web_search(query: str) -> str:
    """Search the public web. Input is a plain search query string."""
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        return f"[FAKE SEARCH] no Tavily key set, pretend results for: {query}"
    r = requests.post(
        "https://api.tavily.com/search",
        json={"api_key": api_key, "query": query, "max_results": 3},
        timeout=15,
    )
    r.raise_for_status()
    data = r.json()
    return "\n".join(f"- {item['title']}: {item['content'][:200]}" for item in data.get("results", []))

tools = [add, multiply, web_search]
tool_registry = {t.name: t for t in tools}

tools_description = "\n".join(f"{t.name}: {t.description}" for t in tools)
tool_names = ", ".join(t.name for t in tools)

print("tools block (this gets pasted into the prompt):")
print(tools_description)
print()
print("tool_names:", tool_names)

## 3. Build the prompt by hand

Let's render the prompt for one example question, with no agent steps yet, so we can see exactly what the LLM receives on the first turn.

In [ ]:
first_turn_prompt = react_prompt.format(
    tools=tools_description,
    tool_names=tool_names,
    input="What is 12 multiplied by 7, then add 100 to that?",
    agent_scratchpad="",
)

print(first_turn_prompt)

Read that. It is a plain string. Nothing fancy. The model will continue this text, starting with `Thought:`.

## 4. Send it to the LLM, stopping at `Observation:`

Critical detail: we tell the LLM to stop generating as soon as it tries to write `Observation:`. Why? Because `Observation` is supposed to come from the tool, not the model. If we let the model continue, it will hallucinate a fake observation.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind(stop=["\nObservation:", "Observation:"])

raw_output = llm.invoke(first_turn_prompt)
print("Raw output from the LLM (first turn):")
print("-" * 60)
print(raw_output.content)
print("-" * 60)

The model wrote a `Thought`, an `Action`, and an `Action Input`, then stopped (because we told it to stop at `Observation:`).

Now we have to parse those two lines out of the text.

## 5. Parse the model's output

This is plain regex on text. There is no JSON. This is what makes ReAct old-school and educational.

In [ ]:
import re

def parse_react_output(text: str):
    """Returns a dict with either ('action', 'action_input') or ('final_answer')."""
    final_match = re.search(r"Final Answer:\s*(.*)", text, re.DOTALL)
    if final_match:
        return {"final_answer": final_match.group(1).strip()}
    action_match = re.search(r"Action:\s*(.*?)\n", text)
    input_match = re.search(r"Action Input:\s*(.*)", text, re.DOTALL)
    if action_match and input_match:
        return {
            "action": action_match.group(1).strip(),
            "action_input": input_match.group(1).strip(),
        }
    raise ValueError(f"Could not parse output:\n{text}")

parsed = parse_react_output(raw_output.content)
print("Parsed:", parsed)

## 6. The full ReAct loop, written from scratch

Now we put it together. Look at how short this is.

In [ ]:
def run_react_agent(question: str, max_iters: int = 8, verbose: bool = True):
    scratchpad = ""
    for step in range(1, max_iters + 1):
        prompt = react_prompt.format(
            tools=tools_description,
            tool_names=tool_names,
            input=question,
            agent_scratchpad=scratchpad,
        )
        ai = llm.invoke(prompt)
        text = ai.content

        if verbose:
            print(f"\n========== iteration {step} ==========")
            print("LLM wrote:")
            print(text)

        try:
            parsed = parse_react_output(text)
        except ValueError as e:
            # Model broke the format. In a real system you'd retry with a hint.
            print(f"\n[parse error] model output did not match expected format: {e}")
            return None

        if "final_answer" in parsed:
            if verbose:
                print("\n>>> DONE. Final answer:")
                print(parsed["final_answer"])
            return parsed["final_answer"]

        action = parsed["action"]
        action_input = parsed["action_input"]

        if action not in tool_registry:
            observation = f"Error: tool '{action}' does not exist. Available: {tool_names}"
        else:
            try:
                observation = str(tool_registry[action].invoke(action_input))
            except Exception as e:
                observation = f"Error running tool: {e}"

        if verbose:
            print(f"\nWe ran tool: {action}({action_input!r})")
            print(f"Observation: {observation}")

        scratchpad += text + f"\nObservation: {observation}\n"

    print(f"\n[max_iters reached] agent did not finish in {max_iters} steps.")
    print("This is a real teaching moment: agents can loop forever if you let them.")
    print("In production you set a budget (max_iters, token cost, wall time) and bail.")
    return None

run_react_agent("What is 12 multiplied by 7, then add 100 to that?")

Read what just happened iteration by iteration. The model:

1. Wrote a thought, picked `multiply`, gave input `12, 7`. We ran it, got `84`, pasted `Observation: 84`.
2. Saw the scratchpad now contains the previous step plus the observation. Wrote a new thought, picked `add`, gave input `84, 100`. We ran it, got `184`, pasted `Observation: 184`.
3. Wrote `Thought: I now know the final answer` and `Final Answer: 184`. We parsed it and stopped.

**That is the entire ReAct loop.** No framework. We built it in about 30 lines of Python.

## 7. Try it on a research question

Now use the search tool. If you have a Tavily key, this will hit the real web.

In [ ]:
run_react_agent("Who is the current CEO of OpenAI, and what year did they take that role?")

## 8. Show what the scratchpad looks like at the end

The scratchpad is the agent's memory. It is the entire transcript of thoughts, actions, and observations so far. On every iteration we send the *same* base prompt with a *bigger* scratchpad.

In [ ]:
def run_and_capture_scratchpad(question, max_iters=6):
    scratchpad = ""
    for step in range(max_iters):
        prompt = react_prompt.format(
            tools=tools_description,
            tool_names=tool_names,
            input=question,
            agent_scratchpad=scratchpad,
        )
        text = llm.invoke(prompt).content
        parsed = parse_react_output(text)
        if "final_answer" in parsed:
            scratchpad += text
            return scratchpad
        obs = str(tool_registry[parsed["action"]].invoke(parsed["action_input"]))
        scratchpad += text + f"\nObservation: {obs}\n"
    return scratchpad

final_scratchpad = run_and_capture_scratchpad("What is 23 * 4, then add 50?")
print("Final scratchpad (the agent's full working memory):")
print("=" * 60)
print(final_scratchpad)

## 9. ReAct vs native tool calling

Both notebooks solve the same problem in different ways. Side by side:

| Aspect | Native tool calling (notebook 1) | ReAct (this notebook) |
|---|---|---|
| How model emits a tool request | Structured `tool_calls` array (JSON) | Plain text following a format |
| How you parse it | Read `response.tool_calls` directly | Regex on the text |
| Failure mode | Schema enforced by the API | Model can break the format and you crash |
| Works on any model | No, needs a model that supports tool calling | Yes, works on any text-completion model |
| Parallel tool calls | Yes, multiple in one response | No, one tool per turn |
| Token usage | Lower | Higher (the format takes tokens) |

Native tool calling is what you should use in production today. ReAct is what you should teach so people understand there is no magic underneath.

## 10. Recap

An agent is:

1. A prompt that tells the model how to format its output (or a model that natively emits structured tool calls).
2. A loop that reads the output, runs the requested tool, appends the result, and calls the model again.
3. An exit condition (`Final Answer:` in ReAct, or no `tool_calls` in the native version).

That is the whole thing. Everything else (LangGraph, AutoGen, CrewAI, OpenAI Swarm) is convenience on top of these three pieces.

**Next:** `05_react_visualized.ipynb` instruments this loop to show token costs and prompt growth at every iteration.